# 10 — Protocol gate before GPU generation

Fail early on duplicate keys, patient leakage, missing references, malformed evidence linkage, or an unbalanced
condition grid. Also report confidence-gate coverage and the number of CXR/ECG queries.

In [ ]:
from pathlib import Path
import os, sys
CODE_ROOT = Path(os.environ.get('EXPERIMENT4_CODE_ROOT', str(Path.cwd())))
if not (CODE_ROOT / 'e4_common.py').is_file() and 'EXPERIMENT4_CODE_ROOT' not in os.environ:
    CODE_ROOT = Path.cwd()
if not CODE_ROOT.is_absolute() or not (CODE_ROOT / 'e4_common.py').is_file():
    raise FileNotFoundError('Set EXPERIMENT4_CODE_ROOT to the uploaded Experiment 4 source directory')
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
from e4_common import ARTIFACT_ROOT, EXPERIMENT4_ROOT, SYMILE_ROOT
print({'code_root': str(CODE_ROOT), 'artifact_root': str(ARTIFACT_ROOT),
       'experiment4_root': str(EXPERIMENT4_ROOT), 'symile_root': str(SYMILE_ROOT)})

In [ ]:
from collections import Counter
from e4_common import read_jsonl
from e4_evidence import validate_contexts
from e4_evaluation import extractor_self_test, reference_audit
rows = read_jsonl(EXPERIMENT4_ROOT / 'contexts' / 'bidirectional_contexts.jsonl')
print(validate_contexts(rows))
keys = [(r['direction'], r['sample_id'], r['method'], r['condition']) for r in rows]
assert len(keys) == len(set(keys)), 'Duplicate context key'
assert all(r['global_patient_disjoint'] for r in rows)
assert all(all(a['linkage'].startswith('same Symile admission') for a in r['evidence']) for r in rows)
counts = Counter((r['direction'], r['method'], r['condition']) for r in rows)
print('rows:', len(rows), 'cells:', len(counts))
print('gate coverage:', {k: sum(r.get('gate_passed') is True for r in rows if r['direction']==k and r['condition']=='full_trimodal_gated') /
      max(sum(r['condition']=='full_trimodal_gated' for r in rows if r['direction']==k),1) for k in ['ecg','cxr']})
print(extractor_self_test())
print(reference_audit())